In [7]:
# Cài đặt ép phiên bản 0.10.14 hoặc 0.10.9
!pip uninstall -y mediapipe
!pip install mediapipe==0.10.14 pandas scikit-learn tensorflow opencv-python

Found existing installation: mediapipe 0.10.33
Uninstalling mediapipe-0.10.33:
  Successfully uninstalled mediapipe-0.10.33
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.7/35.7 MB 66.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 kB 35.2 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.6
    Uninstalling protobuf-5.29.6:
      Successfully uninstalled protobuf-5.29.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grain 0.2.16 requires protobuf>=5.28.3, but you have protobuf 4.25.9 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 4.25.9 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 4.25.9 which is incompatible.
opentelemetry-proto 1.38.0 requires protobuf<7.0,>=5.0, but you have protobuf 4.2

In [2]:
import kagglehub
import os

print("Đang tải dataset...")
dataset_path = kagglehub.dataset_download("akashrayhan/yoga-82")
print("Đã tải xong! Đường dẫn:", dataset_path)

# (Tùy chọn) In ra danh sách các thư mục (các dáng yoga) để kiểm tra
# LƯU Ý: Nếu cấu trúc file của dataset nằm trong thư mục con, bạn cần trỏ đúng path
# Giả sử các ảnh nằm trong các thư mục con mang tên dáng Yoga
print(os.listdir(dataset_path)[:10]) # Xem thử 10 dáng đầu tiên

Đang tải dataset...
Using Colab cache for faster access to the 'yoga-82' dataset.
Đã tải xong! Đường dẫn: /kaggle/input/yoga-82
['valid', 'test', 'train']


In [5]:
import cv2
import mediapipe as mp
import pandas as pd
import numpy as np
import os
import csv
from pathlib import Path

# 1. Khởi tạo MediaPipe
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(static_image_mode=True, min_detection_confidence=0.5)

csv_file = 'yoga_keypoints.csv'
count = 0

# 2. Xóa file cũ nếu có để tránh ghi đè lỗi
if os.path.exists(csv_file):
    os.remove(csv_file)

print(f"Đang quét dữ liệu tại: {dataset_path}")

with open(csv_file, mode='w', newline='') as f:
    writer = csv.writer(f)
    # Header
    header = ['label']
    for i in range(33):
        header.extend([f'x{i}', f'y{i}', f'z{i}', f'v{i}'])
    writer.writerow(header)

    # Dùng rglob để tìm TẤT CẢ ảnh trong mọi thư mục con
    image_paths = list(Path(dataset_path).rglob("*.jpg")) + \
                  list(Path(dataset_path).rglob("*.jpeg")) + \
                  list(Path(dataset_path).rglob("*.png"))

    if len(image_paths) == 0:
        print("❌ KHÔNG TÌM THẤY ẢNH NÀO! Kiểm tra lại dataset_path.")
    else:
        print(f"Tìm thấy {len(image_paths)} ảnh. Đang trích xuất tọa độ...")

    for img_path in image_paths:
        # Lấy tên thư mục chứa ảnh làm Label (Tên dáng Yoga)
        label = img_path.parent.name

        image = cv2.imread(str(img_path))
        if image is None: continue

        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        results = pose.process(image_rgb)

        if results.pose_landmarks:
            row = [label]
            for landmark in results.pose_landmarks.landmark:
                row.extend([landmark.x, landmark.y, landmark.z, landmark.visibility])
            writer.writerow(row)
            count += 1

            # In tiến độ mỗi 100 ảnh
            if count % 100 == 0:
                print(f"✅ Đã xử lý thành công {count} ảnh...")

print(f"\n➡️ HOÀN THÀNH! Đã lưu {count} mẫu vào {csv_file}")

Đang quét dữ liệu tại: /kaggle/input/yoga-82
Tìm thấy 16767 ảnh. Đang trích xuất tọa độ...


/usr/local/lib/python3.12/dist-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


✅ Đã xử lý thành công 100 ảnh...
✅ Đã xử lý thành công 200 ảnh...
✅ Đã xử lý thành công 300 ảnh...
✅ Đã xử lý thành công 400 ảnh...
✅ Đã xử lý thành công 500 ảnh...
✅ Đã xử lý thành công 600 ảnh...
✅ Đã xử lý thành công 700 ảnh...
✅ Đã xử lý thành công 800 ảnh...
✅ Đã xử lý thành công 900 ảnh...
✅ Đã xử lý thành công 1000 ảnh...
✅ Đã xử lý thành công 1100 ảnh...
✅ Đã xử lý thành công 1200 ảnh...
✅ Đã xử lý thành công 1300 ảnh...
✅ Đã xử lý thành công 1400 ảnh...
✅ Đã xử lý thành công 1500 ảnh...
✅ Đã xử lý thành công 1600 ảnh...
✅ Đã xử lý thành công 1700 ảnh...
✅ Đã xử lý thành công 1800 ảnh...
✅ Đã xử lý thành công 1900 ảnh...
✅ Đã xử lý thành công 2000 ảnh...
✅ Đã xử lý thành công 2100 ảnh...
✅ Đã xử lý thành công 2200 ảnh...
✅ Đã xử lý thành công 2300 ảnh...
✅ Đã xử lý thành công 2400 ảnh...
✅ Đã xử lý thành công 2500 ảnh...
✅ Đã xử lý thành công 2600 ảnh...
✅ Đã xử lý thành công 2700 ảnh...
✅ Đã xử lý thành công 2800 ảnh...
✅ Đã xử lý thành công 2900 ảnh...
✅ Đã xử lý thành công 3

In [6]:
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import pickle

# 1. Đọc dữ liệu
df = pd.read_csv('yoga_keypoints.csv')

# Lấy Features (Tọa độ) và Labels (Tên dáng)
X = df.drop('label', axis=1).values
y_labels = df['label'].values

# 2. Chuyển Tên dáng (String) thành Số (Integer)
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_labels)

# Lưu lại Label Encoder để sau này dùng trên Backend map từ số -> chữ
with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(label_encoder, f)

num_classes = len(np.unique(y))
y_categorical = tf.keras.utils.to_categorical(y, num_classes=num_classes)

# 3. Chia tập Train và Test (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y_categorical, test_size=0.2, random_state=42)

# 4. Xây dựng Model DNN đơn giản
model = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=(132,)), # 33 điểm * 4 giá trị (x,y,z,v) = 132
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.3), # Chống overfitting
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(num_classes, activation='softmax') # Đầu ra là số lượng dáng
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# 5. Bắt đầu Train
print("Bắt đầu huấn luyện...")
history = model.fit(X_train, y_train, epochs=50, batch_size=32, validation_data=(X_test, y_test))

# 6. Đánh giá và Lưu Model
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Độ chính xác trên tập test: {accuracy * 100:.2f}%")

# Lưu model định dạng chuẩn
model.save('yoga_pose_model.h5')
print("Đã lưu model tại yoga_pose_model.h5")

Bắt đầu huấn luyện...
Epoch 1/50
372/372 ━━━━━━━━━━━━━━━━━━━━ 9s 15ms/step - accuracy: 0.0924 - loss: 3.9157 - val_accuracy: 0.2486 - val_loss: 3.1486
Epoch 2/50
372/372 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.2352 - loss: 2.9608 - val_accuracy: 0.3787 - val_loss: 2.4062
Epoch 3/50
372/372 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.3232 - loss: 2.4994 - val_accuracy: 0.4857 - val_loss: 1.9907
Epoch 4/50
372/372 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.3946 - loss: 2.1926 - val_accuracy: 0.5483 - val_loss: 1.7604
Epoch 5/50
372/372 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.4504 - loss: 2.0238 - val_accuracy: 0.5651 - val_loss: 1.6338
Epoch 6/50
372/372 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.4688 - loss: 1.9115 - val_accuracy: 0.5930 - val_loss: 1.5386
Epoch 7/50
372/372 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.4992 - loss: 1.8194 - val_accuracy: 0.6347 - val_loss: 1.4052
Epoch 8/50
372/372 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.5188 - loss: 1.

Độ chính xác trên tập test: 77.67%
Đã lưu model tại yoga_pose_model.h5
